<center><h1 style="margin-bottom: 0px;">Artificial Intelligence (25/26)</h1></center>
<center><h2 style="margin-top: 0px;">Pop Out Dataset Generation</h2></center>

#### <center> Joana Antunes (202405702), Sílvia Pinto (202405988) </center> <br>

#### **7. Dataset Generation** <br>
<div style="text-align: justify;"><p style="text-indent: 2em;">To apply our Decision Tree with ID3 model to the Pop Out Game, it is necessary to produce a dataset of the game's states and corresponding moves. The dataset is generated by allowing a Monte Carlo-based agent to play games automatically. At each turn, we store the current game state and the move selected by the algorithm. Each row therefore represents a supervised learning example: the input is each cell in the board state and the target label is the move chosen by the search algorithm.</div>

In [1]:
%run MCTS.ipynb
%run PopOut.ipynb
%run DecisionTrees.ipynb
from multiprocessing import Pool
import pandas as pd
import numpy as np
from collections import Counter
import csv

def generate_dataset(game, games=100, simulations = 100, filename="game_data.csv"):
    dataset = []
    for i in range(games):
        clone = game.clone()
        count = 0
        while not clone.terminal:
            count += 1
            best_move = mcts_move(clone, simulations, "memory+heuristic MCTS")
            cells = []
            for row in range(6):
                for col in range(7):
                    cells.append(clone.board[row][col])
            dataset.append(cells + [f"{best_move[0]}{best_move[1]}"])
            clone.apply_move(best_move, printing = False)
        print(f"Games completed: {i+1}/{games}; added {count} lines.")
    return dataset

In [2]:
if __name__ == "__main__":
    game = Pop_Out()
    dataset = generate_dataset(game, 1, 1000)
    filename = "game_data.csv"
    with open(filename, 'a', newline='') as f:
        writer = csv.writer(f)
        #cell_names = [f"{row}:{col}" for row in range(6) for col in range(7)]
        #writer.writerow(cell_names + ["move"])
        writer.writerows(dataset)
    print(f"Done! Saved {len(dataset)} lines to {filename}")

Games completed: 1/1; added 14 lines.
Done! Saved 14 lines to game_data.csv


Now, we will implement the Decision Tree to our game dataset.

In [3]:
df = pd.read_csv("game_data.csv")

np.random.seed(42)
idx = np.random.permutation(len(df))
split = int(0.8 * len(idx))
train_idx, test_idx = idx[:split], idx[split:] # Train/test split (80/20)

cols = df.columns[:-1]
train_data   = df.iloc[train_idx][cols].to_dict(orient='records')
train_labels = df.iloc[train_idx]['move'].tolist()
test_data    = df.iloc[test_idx][cols].to_dict(orient='records')
test_labels  = df.iloc[test_idx]['move'].tolist()

print(f"Train: {len(train_data)} samples | Test: {len(test_data)} samples")

popout_tree = id3(train_data, train_labels, set(cols)) # Train the tree

default = Counter(train_labels).most_common(1)[0][0]
predictions = [classify(popout_tree, s, default) for s in test_data]
accuracy = sum(p == t for p, t in zip(predictions, test_labels)) / len(test_labels) # Evaluate accuracy
print(f"Accuracy on test set: {accuracy:.2%}")

Train: 872 samples | Test: 218 samples
Accuracy on test set: 69.27%
